# 03 — Run experiments

Configure and launch experiments from a **single configuration cell** below.

Each run writes under the configured storage root (`outputs/runs/` locally, `UH_CV/runs/` on Google Drive).

**Prerequisite:** notebook 02 → `data/annotations/validation_lengths.csv`

In [3]:
from src.colab_bootstrap import setup_notebook_environment
from src.config import get_config
from src.experiments import load_validation_image_ids, run_experiments, results_to_dataframe
from src.experiments.notebook_helpers import (
    RunExperimentsConfig,
    build_experiment_specs,
    preview_experiment_specs,
    project_config_for_experiments,
    resolve_runs_root,
)
from src.utils import setup_logging

REPO, STORAGE = setup_notebook_environment(mount_drive=True)
setup_logging()
cfg = get_config()
print(f"Storage: {STORAGE} | Colab: {cfg.is_colab}")
VALIDATION_IDS = load_validation_image_ids(cfg)
print(f"Validation set: {len(VALIDATION_IDS)} images")

2026-06-02 14:18:31 | INFO     | src.experiments | Loaded 30 validation image IDs from validation_lengths.csv
Validation set: 30 images


## Configuration

Edit this cell only. Set `RUN = True` to execute; leave `False` to preview the experiment grid without running.

Use `experiments=[...]` for explicit runs (legacy mode). Otherwise a grid is built from `pipelines` × `methods` × `splits`.

In [6]:
RUN = True  # set False to preview specs without executing

RUN_CFG = RunExperimentsConfig(
    # --- scope ---
    pipelines=["baseline", "advanced"],           # "baseline" | "advanced"
    methods=["bbox", "pca", "skeleton"],
    splits=["valid"],
    image_ids=None,                    # explicit list overrides validation_set_only
    validation_set_only=True,          # use all IDs from validation_lengths.csv
    limit=None,

    # --- execution ---
    overwrite=True,
    visualize=True,                   # save debug figures under run_dir/figures/
    evaluate=True,
    runs_root=None,                   # default: cfg.runs_root (Drive or outputs/runs)
    run_name_template="{pipeline}_{method}_v1",

    # --- Colab / Drive persistence (all default True) ---
    cache_results=True,
    save_figures_to_drive=True,
    save_metrics_to_drive=True,
    save_predictions_to_drive=True,

    # --- advanced pipeline flags (when pipeline="advanced") ---
    perspective=False,
    use_grid_auto_calibration=False,
    use_depth_estimation=False,
    use_3d_measurement=False,

    # --- optional: explicit experiment list (overrides grid above) ---
    experiments=None,
    # experiments=[
    #     {"pipeline": "advanced", "method": "skeleton", "run_name": "advanced_full_v2",
    #      "split": "valid", "validation_set_only": True, "overwrite": True,
    #      "use_grid_auto_calibration": True, "use_depth_estimation": True,
    #      "use_3d_measurement": True},
    # ],
)

SPECS = build_experiment_specs(RUN_CFG)
preview_experiment_specs(SPECS)

,run_name,pipeline,method,split,visualize,perspective,grid,depth,3d
0,baseline_bbox_v1,baseline,bbox,valid,True,False,False,False,False
1,baseline_pca_v1,baseline,pca,valid,True,False,False,False,False
2,baseline_skeleton_v1,baseline,skeleton,valid,True,False,False,False,False
3,advanced_bbox_v1,advanced,bbox,valid,True,False,False,False,False
4,advanced_pca_v1,advanced,pca,valid,True,False,False,False,False
5,advanced_skeleton_v1,advanced,skeleton,valid,True,False,False,False,False


## Run experiments

In [ ]:
import pandas as pd

exp_cfg = project_config_for_experiments(RUN_CFG)
runs_root = resolve_runs_root(RUN_CFG)

if RUN:
    results = run_experiments(SPECS, cfg=exp_cfg, stop_on_error=RUN_CFG.stop_on_error)
    summary = results
else:
    print("RUN=False — skipped execution. Set RUN=True in the configuration cell.")
    summary = pd.DataFrame()

if len(summary):
    display(summary[[c for c in ["run_name", "pipeline", "method", "n_predictions", "mae_mm", "rmse_mm", "run_dir"] if c in summary.columns]])

2026-06-02 15:41:59 | INFO     | src.experiments | Running experiment 1/6: {'pipeline': 'baseline', 'method': 'bbox', 'split': 'valid', 'run_name': 'baseline_bbox_v1', 'overwrite': True, 'visualize': True, 'evaluate': True, 'limit': None, 'validation_set_only': True}
2026-06-02 15:41:59 | INFO     | src.experiments | Loaded 30 validation image IDs from validation_lengths.csv
2026-06-02 15:41:59 | WARNING  | src.experiments.run_manager | Overwriting existing run directory: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/baseline_bbox_v1
2026-06-02 15:41:59 | INFO     | src.experiments.run_manager | Saved run config: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/baseline_bbox_v1/config.json
2026-06-02 15:41:59 | INFO     | src.pipelines.baseline | BaselinePipeline: split=valid method=bbox perspective=False n_images=30


Measuring (valid, n=30): 0img [00:00, ?img/s]

2026-06-02 15:41:59 | INFO     | src.calibration.marker | Estimated scale: 5.5492 px/mm (n=2 markers, mean)
2026-06-02 15:42:01 | INFO     | src.visualization | Saved figure: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/baseline_bbox_v1/figures/10786_mask.png
2026-06-02 15:42:02 | INFO     | src.visualization | Saved figure: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/baseline_bbox_v1/figures/10786_pca.png


Measuring (valid, n=30): 1img [00:02,  2.11s/img]

2026-06-02 15:42:02 | INFO     | src.calibration.marker | Estimated scale: 2.5409 px/mm (n=4 markers, mean)
2026-06-02 15:42:02 | INFO     | src.visualization | Saved figure: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/baseline_bbox_v1/figures/11050_mask.png
2026-06-02 15:42:02 | INFO     | src.visualization | Saved figure: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/baseline_bbox_v1/figures/11050_pca.png


...

2026-06-02 15:43:27 | INFO     | src.calibration.marker | Estimated scale: 4.0118 px/mm (n=4 markers, mean)
2026-06-02 15:43:31 | INFO     | src.visualization | Saved figure: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/baseline_skeleton_v1/figures/21625_mask.png
2026-06-02 15:43:33 | INFO     | src.visualization | Saved figure: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/baseline_skeleton_v1/figures/21625_pca.png


Measuring (valid, n=30): 13img [00:32,  2.95s/img]

2026-06-02 15:43:33 | INFO     | src.calibration.marker | Estimated scale: 5.3950 px/mm (n=4 markers, mean)
2026-06-02 15:43:40 | INFO     | src.visualization | Saved figure: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/baseline_skeleton_v1/figures/21722_mask.png
2026-06-02 15:43:41 | INFO     | src.visualization | Saved figure: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/baseline_skeleton_v1/figures/21722_pca.png


Measuring (valid, n=30): 14img [00:41,  4.62s/img]

2026-06-02 15:43:41 | INFO     | src.calibration.marker | Estimated scale: 5.2814 px/mm (n=2 markers, mean)


## Sanity check

Confirms every annotated fish was predicted and evaluated when using the validation set.

In [ ]:
if len(summary) and RUN_CFG.validation_set_only and RUN_CFG.image_ids is None:
    n_gt = len(VALIDATION_IDS)
    for _, row in summary.iterrows():
        pred_path = Path(row["predictions_path"])
        comp_path = Path(row["comparison_path"]) if pd.notna(row.get("comparison_path")) else None
        pred = pd.read_csv(pred_path)
        comp = pd.read_csv(comp_path) if comp_path and comp_path.is_file() else None
        print(f"{row['run_name']}: predictions={len(pred)}", end="")
        if comp is not None:
            print(f"  evaluated={len(comp)}", end="")
            assert len(pred) == n_gt, f"{row['run_name']}: not all validation IDs predicted"
            assert len(comp) == n_gt, f"{row['run_name']}: evaluation incomplete"
            print("  OK")
        else:
            print("  (no comparison.csv)")
else:
    print("Skipped — custom image_ids or no runs executed.")